In [0]:
# ============================================================
# Medallion Architecture — Bronze → Silver + Quarantine
# lms_course_master Pipeline
# ============================================================

# COMMAND ----------
# STEP 1: Import Components and Establish Environment Context

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Medallion_Architecture_CourseMaster_Silver_Quarantine") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# Ensure target schemas exist
spark.sql("CREATE SCHEMA IF NOT EXISTS hackathon_ltm.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS hackathon_ltm.quarantine")

print("Target schemas ('silver' and 'quarantine') verified.")


# COMMAND ----------
# STEP 2: Ingest Raw Records from the Bronze Delta Table

bronze_table_name = "hackathon_ltm.bronze.lms_course_master"
df_bronze = spark.read.table(bronze_table_name)

bronze_count = df_bronze.count()
print(f"Bronze records ingested: {bronze_count}")


# COMMAND ----------
# STEP 3: Silver Layer Transformations (Cleansing & Standardization)
#
# TRANSFORMATION 1 — Trim leading/trailing whitespace from all string columns
#   Why: Rows were found with spaces in course_id (" C1151_MS ") and
#        course_name ("Azure Data Fundamentals DP-900 Part 3 ").
#        Untrimmed values cause silent join failures in Gold layer queries.

string_cols = [field.name for field in df_bronze.schema.fields if str(field.dataType) == "StringType()"]
df_transformed = df_bronze
for col_name in string_cols:
    df_transformed = df_transformed.withColumn(col_name, F.trim(F.col(col_name)))

# TRANSFORMATION 2 — Uppercase course_id
#   Why: Row 499 had course_id "c1155_ms" (fully lowercase).
#        course_id is a primary key — its format must be consistent
#        for reliable joins with sp_certification and other tables.

df_transformed = df_transformed.withColumn(
    "course_id",
    F.upper(F.col("course_id"))
)

# TRANSFORMATION 3 — Title Case course_name
#   Why: Row 496 had "azure data fundamentals dp-900 part 2" (fully lowercase).
#        Course names appear on Power BI dashboards and reports —
#        inconsistent casing creates duplicate entries in visuals.

df_transformed = df_transformed.withColumn(
    "course_name",
    F.initcap(F.col("course_name"))
)


# COMMAND ----------
# STEP 4: Duplicate Detection & Separation
#
# LOGIC:
#   A duplicate is defined as two or more rows sharing the same course_id.
#   8 such rows were identified in the dataset.
#
#   Strategy — keep the FIRST occurrence, quarantine the rest:
#   - Within each course_id group, rows are ranked by their original
#     row position (using a monotonically increasing row id assigned
#     before transformation so the ranking is stable).
#   - Rank 1 → clean candidate for Silver.
#   - Rank 2+ → duplicate, tagged with an error and sent to Quarantine.

# Assign a stable row number before deduplication
df_transformed = df_transformed.withColumn("_row_id", F.monotonically_increasing_id())

window_spec = Window \
    .partitionBy("course_id") \
    .orderBy(F.col("_row_id").asc())

df_ranked = df_transformed.withColumn("_row_rank", F.row_number().over(window_spec))

# Rank 1 → moves forward as a clean candidate
df_deduped    = df_ranked.filter(F.col("_row_rank") == 1).drop("_row_rank", "_row_id")

# Rank 2+ → duplicate, will go to quarantine
df_duplicates = df_ranked.filter(F.col("_row_rank") > 1).drop("_row_rank", "_row_id")

duplicate_count = df_duplicates.count()
print(f"Duplicate rows detected and separated: {duplicate_count}")


# COMMAND ----------
# STEP 5: Quarantine Rule Evaluation & Constraint Mapping
#
# Rules are applied to the de-duplicated data only.
#
#   RULE 1 — Missing Primary Key
#     course_id is null or empty after trimming.
#     A row without a course_id cannot be referenced by any other table.
#
#   RULE 2 — Missing course_name
#     course_name is null or empty after trimming.
#     A course without a name is unusable in reports and dashboards.
#
#   RULE 3 — Invalid duration_hours
#     duration_hours is null, zero, or negative.
#     A course must have a positive, non-zero duration.

df_evaluated = df_deduped.withColumn(
    "_failed_constraints",
    F.array(
        # Rule 1 — Missing Primary Key
        F.when(
            F.col("course_id").isNull() | (F.col("course_id") == ""),
            "ERR: Missing Primary Key (course_id)"
        ),

        # Rule 2 — Missing course_name
        F.when(
            F.col("course_name").isNull() | (F.col("course_name") == ""),
            "ERR: Missing course_name"
        ),

        # Rule 3 — Invalid duration_hours
        F.when(
            F.col("duration_hours").isNull() | (F.col("duration_hours") <= 0),
            "ERR: Invalid duration_hours (null, zero, or negative)"
        )
    )
)

# array_compact strips NULL elements — only real error strings remain
# Clean rows  → [] (size 0) → Silver
# Dirty rows  → ["ERR: ..."] (size > 0) → Quarantine
df_evaluated = df_evaluated.withColumn(
    "_failed_constraints",
    F.array_compact(F.col("_failed_constraints"))
    # Spark < 3.4 alternative:
    # F.expr("filter(_failed_constraints, x -> x is not null)")
)


# COMMAND ----------
# STEP 6: Split Data Streams — Clean vs. Constraint-Failed vs. Duplicates

# 6.1 Rows that failed one or more constraint rules → Quarantine
df_quarantine_constraints = df_evaluated.filter(F.size(F.col("_failed_constraints")) > 0)

# 6.2 Clean rows → Silver (drop the internal evaluation column)
df_silver_batch = df_evaluated \
    .filter(F.size(F.col("_failed_constraints")) == 0) \
    .drop("_failed_constraints")

# 6.3 Tag duplicate rows with their error label
df_duplicates_tagged = df_duplicates.withColumn(
    "_failed_constraints",
    F.array(F.lit("ERR: Duplicate course_id"))
)

# 6.4 Union constraint failures + duplicates into one quarantine batch
df_quarantine_batch = df_quarantine_constraints.unionByName(df_duplicates_tagged)


# COMMAND ----------
# STEP 7: Diagnostics — Confirm the split before writing

silver_count             = df_silver_batch.count()
quarantine_constraints_c = df_quarantine_constraints.count()
quarantine_count         = df_quarantine_batch.count()

print(f"\n{'='*55}")
print(f"  Bronze  (total)              : {bronze_count}")
print(f"  Silver  (clean)              : {silver_count}")
print(f"  Quarantine (constraint fail) : {quarantine_constraints_c}")
print(f"  Quarantine (duplicates)      : {duplicate_count}")
print(f"  Quarantine (total)           : {quarantine_count}")
print(f"  Accounted for                : {silver_count + quarantine_count} / {bronze_count}")
print(f"{'='*55}\n")

# Preview the error type breakdown in quarantine
print("Quarantine error distribution:")
df_quarantine_batch \
    .select(F.explode(F.col("_failed_constraints")).alias("error")) \
    .groupBy("error") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show(truncate=False)

# Quick preview of silver
print("Silver sample (5 rows):")
df_silver_batch.show(5, truncate=False)


# COMMAND ----------
# STEP 8: Write to Target Delta Tables

# 8.1 Write clean records to Silver
silver_table_target = "hackathon_ltm.silver.lms_course_master"

df_silver_batch.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(silver_table_target)

print(f"✅ Silver table written → {silver_table_target} ({silver_count} rows)")

# 8.2 Write all quarantine records to Quarantine
quarantine_table_target = "hackathon_ltm.quarantine.lms_course_master"

df_quarantine_batch.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(quarantine_table_target)

print(f"✅ Quarantine table written → {quarantine_table_target} ({quarantine_count} rows)")
print(f"\nPipeline complete.")
print(f"  Clean     → {silver_table_target}")
print(f"  Quarantine → {quarantine_table_target}")

Target schemas ('silver' and 'quarantine') verified.
Bronze records ingested: 500
Duplicate rows detected and separated: 10

  Bronze  (total)              : 500
  Silver  (clean)              : 490
  Quarantine (constraint fail) : 0
  Quarantine (duplicates)      : 10
  Quarantine (total)           : 10
  Accounted for                : 500 / 500

Quarantine error distribution:
+------------------------+-----+
|error                   |count|
+------------------------+-----+
|ERR: Duplicate course_id|10   |
+------------------------+-----+

Silver sample (5 rows):
+---------+----------------------------------------+--------+----------+-------------+--------------+----------------+--------------------------+--------------+---------------------------------------------------------+
|course_id|course_name                             |category|vendor    |delivery_mode|duration_hours|difficulty_level|_ingestion_timestamp      |_source_system|_source_file                                      